In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import yaml

import sys
sys.path.append('..')
from mlrose_hiive import TSPGenerator
from mlrose_hiive import RHCRunner

In [2]:
SEED = 0
RESTART_LIST = [0, 25, 50, 75, 100]

PROBLEM_SIZE_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['problem_size']]
ITERATIONS_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['iterations']]
MAX_ATTEMPTS_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['attempts']]
NUM_RUNS = yaml.load(open('parameters.yml'), yaml.Loader)['num_runs']
POPULATION_SIZE_LIST = [yaml.load(open('parameters.yml'), yaml.Loader)['population_size']]

assert(len(PROBLEM_SIZE_LIST) == 1)
EXPERIMENT_NAME = f'TSP_RHC'

In [3]:
output_dir = f'metrics_size={PROBLEM_SIZE_LIST[0]}_iters={ITERATIONS_LIST[0]}_pop={POPULATION_SIZE_LIST[0]}_attempts={MAX_ATTEMPTS_LIST[0]}/{EXPERIMENT_NAME}'
os.makedirs(output_dir, exist_ok=True)

In [4]:
all_df = pd.DataFrame()
group_i = 0
run_i = 0
for problem_size in PROBLEM_SIZE_LIST:
    for iterations in ITERATIONS_LIST:
        for max_attempts in MAX_ATTEMPTS_LIST:
            for restarts in RESTART_LIST:
                for i in tqdm(range(NUM_RUNS)):
                    problem = TSPGenerator().generate(seed=SEED+i, number_of_cities=problem_size)
                    sa = RHCRunner(
                        problem=problem,
                        experiment_name='dontcare',
                        output_directory='./',
                        seed=SEED,
                        max_attempts=max_attempts,
                        iteration_list=[iterations],
                        restart_list=[restarts],
                    )
                    _, df_run_curves = sa.run()
                    df_run_curves['group_number'] = group_i
                    df_run_curves['run_number'] = run_i
                    df_run_curves['problem_size'] = problem_size
                    df_run_curves['max_iterations'] = iterations
                    df_run_curves['max_attempts'] = max_attempts
                    df_run_curves['max_restarts'] = restarts

                    print(f"Max Fitness: {df_run_curves['Fitness'].max()}")
                    print(f"Max Iteration: {df_run_curves['Iteration'].max()}")

                    all_df = pd.concat([all_df, df_run_curves], axis=0)
                    run_i += 1
                group_i += 1
all_df.reset_index(inplace=True, drop=True)

100%|██████████| 3/3 [00:00<00:00, 131.04it/s]


Max Fitness: 1298.551899183571
Max Iteration: 10
Max Fitness: 1417.3575899077787
Max Iteration: 10
Max Fitness: 1433.26174126467
Max Iteration: 10


 33%|███▎      | 1/3 [00:00<00:00,  7.89it/s]

Max Fitness: 1657.5787544570599
Max Iteration: 10


 67%|██████▋   | 2/3 [00:00<00:00,  8.05it/s]

...

Max Fitness: 1701.9904989316822
Max Iteration: 10


 67%|██████▋   | 2/3 [00:02<00:01,  1.04s/it]

Max Fitness: 1467.1002865844316
Max Iteration: 10


100%|██████████| 3/3 [00:02<00:00,  1.04it/s]

Max Fitness: 1716.2626092414482
Max Iteration: 10


In [5]:
all_df['Fitness'].min(), all_df['Fitness'].max()

(809.2224969730867, 1716.2626092414482)

In [6]:
all_df.to_csv(os.path.join(output_dir, 'learning_curve.csv'), index=False)